In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator , TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from imblearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, cross_validate , train_test_split
from dataclasses import dataclass
from sklearn.model_selection import RandomizedSearchCV , GridSearchCV

from xgboost import XGBClassifier
import scipy

ModuleNotFoundError: No module named 'MlUtlitys'

In [ ]:
linux_path = r"/run/media/drdrakken/Elements/Sonstiges/Programmieren/Machine Learning/csvs/BankChurn/archive.zip"
df = pd.read_csv(linux_path)

In [ ]:
df

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0,0,1,DIAMOND,300
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0,0,5,PLATINUM,771
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1,1,3,SILVER,564
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1,1,2,GOLD,339


In [ ]:
pd.crosstab(
    [df["Gender"], df["NumOfProducts"]],
    df["Exited"],
    normalize="index"
)

Exited                       0         1
Gender NumOfProducts                    
Female 1              0.668118  0.331882
       2              0.898058  0.101942
       3              0.134228  0.865772
       4              0.000000  1.000000
Male   1              0.767934  0.232066
       2              0.945059  0.054941
       3              0.222222  0.777778
       4              0.000000  1.000000

In [ ]:
@dataclass
class Config():
    target : str = "Exited"
    seed : int = 1234
    test_size : float = 0.2
    cross_iterations : int = 2
    verbose : int = 0

    training_results_output_save_dir = r"/run/media/drdrakken/Elements/Sonstiges/Programmieren/Machine Learning/Part 13 - ProjectFolder/MLModels/BankChurn/"
    training_results_output_save_name : str = "bank_churn_results.csv"
    training_results_output_save_path : str = training_results_output_save_dir + training_results_output_save_name

    csv_save_dir : str = r"/run/media/drdrakken/Elements/Sonstiges/Programmieren/Machine Learning/Part 13 - ProjectFolder/MLModels/BankChurn/"
    csv_save_name :str = "cv_results.csv"
    csv_save_path : str = csv_save_dir + csv_save_name
config = Config()

In [ ]:
class DataClass():

    def __init__(self):
        self.orig_data = df.copy()

        self.x = self.orig_data.drop([config.target] , axis = 1)
        self.y = self.orig_data[config.target]

        self.numerical_data = self.x.select_dtypes(include = np.number).columns
        self.categorical_data = self.x.select_dtypes(exclude = np.number).columns
data = DataClass()

In [ ]:
class Visualize():

    def __init__(self):
        self.data = data
        self.plot()

    def iqr(self , TargetCol):
        q1 = self.data.x[TargetCol].quantile(0.25)
        q3 = self.data.x[TargetCol].quantile(0.75)
        iqr = q3 - q1 
        return q1 , q3 , iqr
    
    def skew(self , SkewCol):
        data_skew = self.data.x[SkewCol].skew()
        print(f"Skewness of {SkewCol}:->:{data_skew}")

    def plot(self):
        for vis in self.data.numerical_data:
            fig , axes = plt.subplots(3 , 1, figsize = (10 , 10 ) , dpi = 200)
            data = self.data.x
            mean = data[vis].mean()
            q1 , q3 , _ = self.iqr(TargetCol =vis)
            self.skew(SkewCol = vis)

            sns.histplot(data = data , x = vis  , ax = axes[0])
            axes[0].axvline(q1 , color = "green")
            axes[0].axvline(q3 , color = "red")
            axes[0].axvline(mean , color = "yellow")
            axes[0].set_title(f"Histplot for{vis}")

            sns.boxplot(data = data , x = vis ,  ax = axes[1])
            axes[1].axvline(q1 , color = "green")
            axes[1].axvline(q3 , color = "red")
            axes[1].axvline(mean , color = "yellow")
            axes[1].set_title(f"Boxplot for{vis}")

            sns.scatterplot(data = data , x = vis  , ax = axes[2])
            axes[2].axvline(q1 , color = "green")
            axes[2].axvline(q3 , color = "red")
            axes[2].axvline(mean , color = "yellow")
            axes[2].set_title(f"Scatterplot for{vis}")

            plt.tight_layout()
            plt.show()
#Visualize()


In [ ]:
class DataTransform(BaseEstimator , TransformerMixin):

    def fit(self , X , y = None):
        return self

    def transform(self, X):
        x = X.copy()
        x = self.drop_features(x)
        x = self.new_features(x)
        return x
    
    def new_features(self , X , y = None):
        x = X.copy()

        return x
    
    def drop_features(self , X):
        x = X.copy()
        dropCols = [
            "RowNumber"	,
            "CustomerId",
            "Surname",
        ]
        for cols in dropCols:
            x = x.drop([cols] , axis = 1)
        return x
        	

In [ ]:
class Preprocess(BaseEstimator , TransformerMixin):

    def fit(self , X , y = None):
        numerical_data = X.select_dtypes(include = np.number).columns
        categorical_data = X.select_dtypes(exclude = np.number).columns
        
        self.preprocess_data = ColumnTransformer([
            ("numerical_data_process" , Pipeline([
                ("imputer" , SimpleImputer(strategy = "mean")),
                ("scale" , MinMaxScaler()),
            ]),numerical_data),

            ("categorical_data_preprocess" , Pipeline([
                ("imputer" , SimpleImputer(strategy = "most_frequent")),
                ("encoder" , OneHotEncoder(handle_unknown = "ignore" , sparse_output= False))
            ]),categorical_data)
        ])
        self.preprocess_data.fit(X)
        return self

    def transform(self, X , y = None):
        return self.preprocess_data.transform(X)

In [ ]:
def model_varianz():
    
    return {    

        "LogisticRegression":LogisticRegression(random_state = config.seed),
        "DecisionTreeClassifier":DecisionTreeClassifier(random_state = config.seed),
        "RandomForestClassifier":RandomForestClassifier(random_state = config.seed),
        "XGBClassifier":XGBClassifier(random_state = config.seed),

    }
        
    


In [ ]:
def model_parameters(model_name):

    return {

        "LogisticRegression": {

            "grid": {
                "estimator__C": [0.01, 0.1, 1, 10, 100],
                "estimator__penalty": ["l2"],
                "estimator__solver": ["lbfgs"],
            },

            "random": {
                "estimator__C": scipy.stats.loguniform(1e-4, 1e2),
                "estimator__penalty": ["l2"],
                "estimator__solver": ["lbfgs"],
            }
        },

        "DecisionTreeClassifier": {

            "grid": {
                "estimator__criterion": ["gini", "entropy"],
                "estimator__max_depth": [None, 5, 10, 20],
                "estimator__min_samples_split": [2, 5, 10],
                "estimator__min_samples_leaf": [1, 2, 4],
            },

            "random": {
                "estimator__criterion": ["gini", "entropy"],
                "estimator__max_depth": scipy.stats.randint(2, 30),
                "estimator__min_samples_split": scipy.stats.randint(2, 20),
                "estimator__min_samples_leaf": scipy.stats.randint(1, 10),
            }
        },

        "RandomForestClassifier": {

            "grid": {
                "estimator__n_estimators": [100, 200, 300],
                "estimator__max_depth": [None, 10, 20, 30],
                "estimator__min_samples_split": [2, 5, 10],
                "estimator__min_samples_leaf": [1, 2, 4],
                "estimator__max_features": ["sqrt", "log2"],
            },

            "random": {
                "estimator__n_estimators": scipy.stats.randint(100, 500),
                "estimator__max_depth": scipy.stats.randint(5, 40),
                "estimator__min_samples_split": scipy.stats.randint(2, 20),
                "estimator__min_samples_leaf": scipy.stats.randint(1, 10),
                "estimator__max_features": ["sqrt", "log2"],
            }
        },

        "XGBClassifier": {

            "grid": {
                "estimator__n_estimators": [100, 200, 300],
                "estimator__learning_rate": [0.01, 0.05, 0.1],
                "estimator__max_depth": [3, 5, 7],
                "estimator__subsample": [0.8, 1.0],
                "estimator__colsample_bytree": [0.8, 1.0],
            },

            "random": {
                "estimator__n_estimators": scipy.stats.randint(100, 500),
                "estimator__learning_rate": scipy.stats.loguniform(1e-3, 0.3),
                "estimator__max_depth": scipy.stats.randint(3, 10),
                "estimator__subsample": scipy.stats.uniform(0.6, 0.4),
                "estimator__colsample_bytree": scipy.stats.uniform(0.6, 0.4),
            }
        }

    }

In [ ]:
def custom_grid_search(estimator , X_train , y_train  , grid_params ,random_params , grid_scoring , random_grid_scoring ):

    grid = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=random_params,
        n_iter=20,
        cv=config.cross_iterations,
        scoring= random_grid_scoring,
        random_state=config.seed,
        n_jobs=-1,
        verbose=config.verbose,
        refit = "roc_auc",
        return_train_score=True,

    )
    grid.fit(X_train , y_train)
    return grid

In [ ]:
def custom_pipeline(estimator , Transform = False):
    steps = []
    
    if Transform:
        steps.append(("Transformed_Data" , DataTransform()))

    steps.append(("BasePerformance" , Preprocess()))
    steps.append(("estimator" , estimator))
    return Pipeline(steps)

In [ ]:
def custom_data_validation(estimator , X_train , y_train, scoring_dict):
    k_split = KFold(n_splits = config.cross_iterations , shuffle = True , random_state = config.seed)

    return cross_validate(estimator = estimator,
                          X= X_train ,
                          y = y_train,
                          cv = k_split,
                          return_estimator= True,
                          return_train_score=True,
                          scoring = scoring_dict,
                          verbose = config.verbose,
                          )

In [ ]:
def custom_paramter_grid():

        return {

            "LogisticRegression": {
                "estimator__C": scipy.stats.loguniform(1e-4, 1e2),
                "estimator__penalty": ["l2"],
                "estimator__solver": ["lbfgs"],
            },

            "DecisionTreeClassifier": {
                "estimator__criterion": ["gini", "entropy"],
                "estimator__max_depth": [None, 5, 10, 20],
                "estimator__min_samples_split": scipy.stats.randint(2, 20),
                "estimator__min_samples_leaf": scipy.stats.randint(1, 10),
            },

            "RandomForestClassifier": {
                "estimator__n_estimators": scipy.stats.randint(100, 500),
                "estimator__max_depth": scipy.stats.randint(5, 40),
                "estimator__min_samples_split": scipy.stats.randint(2, 20),
                "estimator__min_samples_leaf": scipy.stats.randint(1, 10),
                "estimator__max_features": ["sqrt", "log2"],
                "estimator__bootstrap": [True, False],
            },

            "LinearSVC": {
                "estimator__C": scipy.stats.loguniform(1e-4, 1e2),
                "estimator__loss": ["hinge", "squared_hinge"],
                "estimator__dual": [True],
                "estimator__max_iter": [5000, 10000],
            },

            "XGBClassifier": {
                "estimator__n_estimators": scipy.stats.randint(100, 500),
                "estimator__learning_rate": scipy.stats.loguniform(1e-3, 0.3),
                "estimator__max_depth": scipy.stats.randint(3, 10),
                "estimator__subsample": scipy.stats.uniform(0.6, 0.4),
                "estimator__colsample_bytree": scipy.stats.uniform(0.6, 0.4),
                "estimator__gamma": scipy.stats.uniform(0, 5),
                "estimator__min_child_weight": scipy.stats.randint(1, 10),
            },

        }


In [ ]:
def custom_scoring_dict():
        return {

            "f1":"f1",
            "accuracy":"accuracy",
            "precision":"precision",

        }


In [ ]:
def plot_scores(self, data, x_column,score_columns,title="Model Comparison",ylabel="Score"):

            if isinstance(data, list):
                data = pd.DataFrame(data)

            x = range(len(data))

            plt.figure(figsize=(12, 6))

            for column in score_columns:
                if column in data.columns:
                    plt.plot(
                        x,
                        data[column],
                        marker="o",
                        linewidth=2,
                        label=column
                    )

            plt.xticks(x, data[x_column], rotation=45)
            plt.xlabel(x_column)
            plt.ylabel(ylabel)
            plt.title(title)

In [ ]:
def split_data():
    X_train , X_test , y_train , y_test = train_test_split(data.x , 
                                                           data.y,
                                                           test_size = config.test_size,
                                                           random_state = config.seed,
                                                           shuffle = True,
                                                           stratify = data.y)
    
    print(f"✓ Data split completed:")
    print(f"  • Training set: {len(X_train)} samples")
    print(f"  • Test set: {len(X_test)} samples")
    print(f"  • Features: {len(data.x.columns)}")
    
    return X_train , X_test , y_train , y_test

In [ ]:
class Benchmark():

    def __init__(self , UseCV = False , UseGrid = False ):
        self.X_train , self.X_test , self.y_train , self.y_test = split_data()
        self.results = []

        self.use_cv = UseCV
        self.use_grid_search = UseGrid

        self.train()

    def train(self):

        for estimator_name , estimators in model_varianz().items():

            base_pipe = custom_pipeline(estimator=estimators , transform= False , smote=False)
            transformed_pipe = custom_pipeline(estimator=estimators , transform= True , smote=True)

            if self.use_cv:
                base_cv = custom_data_validation(estimator=base_pipe,
                                                X_train=self.X_train,
                                                y_train=self.y_train,
                                                scoring_dict=custom_scoring_dict())
                
                transformed_cv = custom_data_validation(estimator=transformed_pipe,
                                                X_train=self.X_train,
                                                y_train=self.y_train,
                                                scoring_dict=custom_scoring_dict())
                
                print(f"keys;{base_cv.keys()}")

                self.results.append({

                    "base_cv_train_performance":base_cv["train_roc_auc"].mean(),
                    "base_cv_test_performance":base_cv["test_roc_auc"].mean(),
                    "base_cv_std_performance":base_cv["test_roc_auc"].std(),

                    "base_cv_train_performance":base_cv["train_f1"].mean(),
                    "base_cv_test_performance":base_cv["test_f1"].mean(),
                    "base_cv_std_performance":base_cv["test_f1"].std(),

                    "base_cv_train_pr_auc_performance":base_cv["train_pr_auc"].mean(),
                    "base_cv_test_pr_auc_performance":base_cv["test_pr_auc"].mean(),
                    "base_cv_test_pr_auc_performance":base_cv["test_pr_auc"].std(),


                    "transformed_cv_train_performance":transformed_cv["train_roc_auc"].mean(),
                    "transformed_cv_test_performance":transformed_cv["test_roc_auc"].mean(),
                    "transformed_cv_std_performance":transformed_cv["test_roc_auc"].std(),

                    "transformed_cv_train_performance":transformed_cv["train_f1"].mean(),
                    "transformed_cv_test_performance":transformed_cv["test_f1"].mean(),
                    "transformed_cv_std_performance":transformed_cv["test_f1"].std(),

                    "transformed_cv_train_pr_auc_performance":transformed_cv["train_pr_auc"].mean(),
                    "transformed_cv_test_pr_auc_performance":transformed_cv["test_pr_auc"].mean(),
                    "transformed_cv_test_pr_auc_performance":transformed_cv["test_pr_auc"].std(),

                })

            if self.use_grid_search:
                base_grid = custom_grid_search(estimator=base_pipe,
                                            X_train=self.X_train,
                                            y_train=self.y_train,
                                            param_grid=custom_paramter_grid()[estimator_name])

                transformed_grid = custom_grid_search(estimator=base_pipe,              
                                            X_train=self.X_train,
                                            y_train=self.y_train,
                                            param_grid=custom_paramter_grid()[estimator_name])
                
            self.results.append({

                "base_cv_train_performance":base_grid.best_estimator_,
                "transformed_cv_train_performance":transformed_grid.best_estimator_,
                
            })

        plot_scores(data=self.results)
        
          

In [ ]:
Benchmark()

✓ Data split completed:
  • Training set: 8000 samples
  • Test set: 2000 samples
  • Features: 17
self.use_cv:True


/home/drdrakken/.clean_venv/lib64/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/drdrakken/.clean_venv/lib64/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/drdrakken/.clean_venv/

TypeError: cannot unpack non-iterable RandomizedSearchCV object